# 08 - MobileNetV2 Setup (E1 preparation only)

Paste the **single complete code cell below** at the bottom of the existing, verified Kaggle notebook and run **only that new cell**. Do not restart, Run All, re-upload a notebook, recopy images or rebuild the transfer bundle. Keep both existing inputs and GPU `cuda:0`. Required state/files are checked explicitly; missing state stops the cell. The official pretrained download requires Kaggle internet access or an already cached matching weight file.

This is our additional architecture choice, not a model specifically taught in the course. Transfer learning is permitted by Task 3 and covered in Lecture 11 (PDF page 24, slide 93). E0 remains unchanged. The only training here is a bounded disposable diagnostic: at most two warm-up, three timed classifier-training and two validation batches, with a 90-second diagnostic budget after setup/download. The cell saves `/kaggle/working/mobilenetv2_setup_check.json`; return its printed final summary and JSON. No full E1 experiment is started.

## 1. Preserve geometry; normalise once

The existing Dataset retains the exact split (50,927/12,733), class encoding and string IDs. Training augmentation remains horizontal flip and mild brightness/contrast, before RGB bilinear longest-side resize and grey centre-padding to 224. The wrapper adds ImageNet channel normalisation once to existing float32 `/255` tensors. This deliberately retains facial-feature-preserving geometry instead of the official V2 resize232/centre-crop224 preset; the official transform is **not** called. Normalised tensors can extend outside [0,1].

## 2. Freeze features and BatchNorm; verify disposable updates

`FrozenMobileNetV2` uses explicit `MobileNet_V2_Weights.IMAGENET1K_V2`, a Dropout(0.3)/Linear(last_channel,8) classifier and raw logits. Its `train()` override keeps features and BatchNorm in eval mode while the classifier trains. Diagnostic weights are discarded. Any later E1 run must reset seed42 and create a fresh pretrained model, optimiser and loaders. Temporary `weights=None` construction is used only as a skeleton immediately loaded strictly for the checkpoint round-trip; local weights=None checks are structural only, never evidence of pretrained/GPU success.

Reference: [official torchvision MobileNetV2 documentation](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v2.html). Local verification covers syntax, structure, normalisation and frozen-feature/BatchNorm behaviour without downloading weights. Hosted pretrained/CUDA verification remains pending.


In [ ]:
# E1 SETUP / DISPOSABLE VERIFICATION ONLY — run this one new cell.
RUN_FULL_TRAINING = False
import hashlib
import json
import platform
import statistics
import sys
import tempfile
import time
from pathlib import Path
from urllib.parse import urlparse
try:
    import numpy as np
    import pandas as pd
    import torch
    import torchvision
    from PIL import __version__ as pillow_version
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
    from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
except ImportError as error:
    failure = {"status": "failed", "checks": {}, "failures": [str(error)]}
    if Path("/kaggle/working").is_dir():
        Path("/kaggle/working/mobilenetv2_setup_check.json").write_text(json.dumps(failure, indent=2))
    raise RuntimeError("Required hosted dependency unavailable; nothing installed") from error


class ImageNetNormalizedDataset(Dataset):
    """Wrap verified /255 tensors; retain IDs, labels, geometry and augmentation."""
    def __init__(self, base):
        self.base = base
        self.mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
        self.std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        image, label, image_id = self.base[index]
        if image.dtype != torch.float32 or image.shape != (3, 224, 224):
            raise ValueError("Expected reference-preprocessed float32 [3,224,224]")
        return (image - self.mean) / self.std, label, image_id  # Normalise once only.


class FrozenMobileNetV2(nn.Module):
    """Official pretrained features with an eight-logit, trainable classifier."""
    def __init__(self, weights=MobileNet_V2_Weights.IMAGENET1K_V2):
        super().__init__()
        if weights is not None and not Path("/kaggle/working").is_dir():
            raise RuntimeError("Pretrained downloads are allowed in Kaggle only")
        self.network = mobilenet_v2(weights=weights)
        self.network.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.network.last_channel, 8))
        for parameter in self.network.features.parameters():
            parameter.requires_grad_(False)
        self.train()

    def train(self, mode=True):
        super().train(mode)
        # Frozen weights alone do not freeze BatchNorm running statistics.
        self.network.features.eval()
        return self

    def forward(self, images):
        return self.network(images)  # Raw logits; no softmax before cross-entropy.


def _verify_mobilenetv2_setup():
    output = Path("/kaggle/working/mobilenetv2_setup_check.json")
    result = {"status": "running", "diagnostic_only": True, "checks": {}, "failures": [],
        "configuration": {"seed": 42, "batch_size": 32, "num_workers": 0, "drop_last": False,
            "training_shuffle": True, "validation_shuffle": False, "device": "cuda:0",
            "dtype": "float32", "amp": False, "multi_gpu": False,
            "optimizer": "Adam(classifier only)", "learning_rate": 0.001,
            "loss": "unweighted CrossEntropyLoss", "diagnostic_budget_seconds": 90},
        "preprocessing": {"base": "unchanged RGB / bilinear longest-side-224 / grey centre-pad /255",
            "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
            "normalisation_applications": 1,
            "adaptation": "Keep reference geometry; do not use official resize232/centre-crop224 preset"},
        "batch_seconds": {"warmup": [], "training": [], "validation": []},
        "completed_batches": {"warmup": 0, "training": 0, "validation": 0},
        "stop_reasons": [], "limitations": [
            "Diagnostic updates are discarded; later E1 training needs a fresh seeded pretrained model.",
            "No E1 experiment, history, learning curves or permanent checkpoint is created.",
            "The 90s diagnostic budget starts after setup, download and snapshots; one batch can overrun.",
            "Short loading-inclusive timings include verification overhead and cache effects.",
            "An epoch projection is rough, not a measured epoch; no GPU bitwise reproducibility claim."]}
    model = optimizer = reloaded = fixed = None
    temporary = None
    started = None

    def check(name, condition):
        result["checks"][name] = bool(condition)
        if not condition:
            raise ValueError(name)

    def digest(path):
        with Path(path).open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    def room(deadline, estimate=0):
        if time.perf_counter() - started + estimate >= deadline:
            result["stop_reasons"].append(f"Stopped before next operation: {deadline}s phase deadline")
            return False
        return True

    try:
        missing = [name for name in ["SETUP_ROOT", "TRAIN_CACHE_DIR", "report", "training_cache_check", "artifact_hashes"]
                   if name not in globals()]
        if missing:
            raise RuntimeError("Missing verified session state: " + ", ".join(missing) + "; no setup/copy/training rerun")
        check("kaggle_working_directory_available", output.parent.is_dir())
        setup, cache = Path(SETUP_ROOT).resolve(), Path(TRAIN_CACHE_DIR).resolve()
        setup_report_path = output.parent / "gpu_setup_check.json"
        cache_report_path = output.parent / "training_image_cache_check.json"
        for path in [setup_report_path, cache_report_path, setup / "transfer_manifest.json"]:
            if not path.is_file():
                raise FileNotFoundError(f"Required verified file missing: {path}")
        setup_disk = json.loads(setup_report_path.read_text())
        cache_disk = json.loads(cache_report_path.read_text())
        check("setup_and_fingerprints_passed", all(item.get("status") == "passed"
            and not item.get("failures") and item.get("checks") and all(item["checks"].values())
            and item.get("gpu_check_executed") and item["checks"].get("all_strict_fingerprints_match")
            for item in [report, setup_disk]))
        check("complete_verified_working_cache", cache == (output.parent / "enge707_train_cache").resolve()
            and cache.is_dir() and all(item.get("status") == "passed" and item.get("verified_files") == 63660
            and not item.get("failures") and not item.get("hash_mismatches")
            and item.get("checks") and all(item["checks"].values())
            and Path(item["cache_directory"]).resolve() == cache for item in [training_cache_check, cache_disk]))
        pinned = "14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
        check("pinned_transfer_manifest", digest(setup / "transfer_manifest.json") == pinned)
        transfer = json.loads((setup / "transfer_manifest.json").read_text())
        expected_hashes = {**transfer["sha256"], "transfer_manifest.json": pinned}
        check("session_hashes_match_pinned_manifest", artifact_hashes == expected_hashes)
        actual_hashes = {name: digest(setup / name) for name in expected_hashes}
        check("all_split_mapping_source_hashes_match", actual_hashes == expected_hashes)
        result["input_sha256"] = {**actual_hashes, "setup_report": digest(setup_report_path),
                                  "cache_report": digest(cache_report_path)}
        module = sys.modules.get("src.data_pipeline")
        check("existing_dataset_module_from_verified_setup", module is not None
              and Path(module.__file__).resolve() == setup / "src/data_pipeline.py")
        from src.data_pipeline import FacialFeatureDataset, seed_everything
        split = pd.read_csv(setup / "data/processed/split_manifest.csv", dtype=str,
                            keep_default_na=False, na_filter=False)
        mapping = json.loads((setup / "data/processed/class_mapping.json").read_text())
        names = sorted(mapping["class_to_index"])
        check("exact_class_mapping", len(names) == 8 and mapping == transfer["classes"]
            and mapping["class_to_index"] == dict(zip(names, range(8)))
            and all(mapping["index_to_class"][str(i)] == name for i, name in enumerate(names)))
        check("saved_split_counts_ids", len(split) == 63660 and split["image_id"].is_unique
            and split["image_id"].str.fullmatch(r"[0-9]{5}").all()
            and split["split"].value_counts().to_dict() == {"train": 50927, "validation": 12733})
        rows = {name: split.loc[split["split"].eq(name)].reset_index(drop=True) for name in ["train", "validation"]}
        overlaps = {key: len(set(rows["train"][key]) & set(rows["validation"][key]))
                    for key in ["image_id", "byte_sha256", "pixel_sha256"]}
        check("duplicate_disjoint_and_all_classes", not any(overlaps.values())
              and all(set(frame["class"]) == set(names) for frame in rows.values()))
        bases = {name: FacialFeatureDataset(setup, name, training_image_directory=cache) for name in rows}
        check("both_datasets_keep_membership_order", all(len(bases[name]) == len(rows[name])
              and bases[name].rows.equals(rows[name][["image_id", "class", "image_path"]]) for name in rows))
        check("training_only_original_augmentation", bases["validation"].augmentation is None
              and bases["train"].augmentation is not None)
        datasets = {name: ImageNetNormalizedDataset(base) for name, base in bases.items()}
        raw, label, image_id = bases["validation"][0]
        first, label1, id1 = datasets["validation"][0]
        repeated, label2, id2 = datasets["validation"][0]
        check("validation_deterministic_correct_normalisation", torch.equal(first, repeated)
            and torch.equal(first, (raw - datasets["validation"].mean) / datasets["validation"].std)
            and torch.allclose(first * datasets["validation"].std + datasets["validation"].mean, raw, atol=1e-7)
            and bool(torch.isfinite(first).all()) and bool((raw >= 0).all() & (raw <= 1).all())
            and label == label1 == label2 and image_id == id1 == id2 == rows["validation"].iloc[0]["image_id"])
        result["dataset_lengths"] = {name: len(data) for name, data in datasets.items()}
        result["class_mapping"], result["overlap_counts"] = mapping, overlaps
        result["training_augmentation"] = repr(bases["train"].augmentation)
        check("cuda_available", torch.cuda.is_available())
        device = torch.device("cuda:0")
        result["runtime"] = {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": torchvision.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "Pillow": pillow_version, "cuda_build": torch.version.cuda}
        result["device"] = {"selected": str(device), "name": torch.cuda.get_device_name(0),
            "free_bytes": torch.cuda.mem_get_info(0)[0], "cpu_threads": torch.get_num_threads()}
        weights = MobileNet_V2_Weights.IMAGENET1K_V2
        check("official_explicit_weights_url", weights.url == "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth")
        weights_path = Path(torch.hub.get_dir()) / "checkpoints" / Path(urlparse(weights.url).path).name
        result["weights"] = {"identifier": "MobileNet_V2_Weights.IMAGENET1K_V2", "url": weights.url,
                             "cache_path": str(weights_path), "already_cached": weights_path.is_file()}
        if weights_path.exists():
            check("cached_weights_hash_prefix", digest(weights_path).startswith("7ebf99e0"))
        seed_everything(42)
        # The official builder downloads these exact weights if absent. Errors propagate.
        model = FrozenMobileNetV2(weights=weights).to(device=device, dtype=torch.float32)
        result["weights"]["sha256"] = digest(weights_path)
        check("downloaded_weights_hash_prefix", result["weights"]["sha256"].startswith("7ebf99e0"))
        result["parameters"] = {"total": sum(p.numel() for p in model.parameters()),
            "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
        check("classifier_only_trainable", all(not p.requires_grad for p in model.network.features.parameters())
            and all(p.requires_grad for p in model.network.classifier.parameters())
            and result["parameters"]["trainable"] == model.network.last_channel * 8 + 8)
        optimizer = torch.optim.Adam(model.network.classifier.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        frozen_before = {name: value.detach().cpu().clone() for name, value in model.network.features.state_dict().items()}
        head_before = {name: value.detach().cpu().clone() for name, value in model.network.classifier.state_dict().items()}
        loaders = {"train": DataLoader(datasets["train"], batch_size=32, shuffle=True,
            generator=torch.Generator().manual_seed(42), num_workers=0, drop_last=False),
            "validation": DataLoader(datasets["validation"], batch_size=32, shuffle=False, num_workers=0, drop_last=False)}
        result["loader_batches"] = {name: len(loader) for name, loader in loaders.items()}
        validation_ids = []
        expected_labels = {name: frame.set_index("image_id")["class"].map(mapping["class_to_index"]).to_dict()
                           for name, frame in rows.items()}
        torch.cuda.synchronize(device)
        started = time.perf_counter()
        training_iterator, validation_iterator = iter(loaders["train"]), iter(loaders["validation"])
        previous = 0.0
        for phase, maximum, deadline in [("warmup", 2, 45), ("training", 3, 60), ("validation", 2, 75)]:
            if phase == "training" and result["completed_batches"]["warmup"] < 2:
                result["stop_reasons"].append("Training timings skipped: warm-ups incomplete")
                continue
            for number in range(maximum):
                if not room(deadline, previous):
                    break
                torch.cuda.synchronize(device)
                batch_started = time.perf_counter()
                training = phase != "validation"
                images, targets, ids = next(training_iterator if training else validation_iterator)
                partition = "train" if training else "validation"
                check("batch_labels_ids_" + phase + str(number), targets.tolist() == [expected_labels[partition][i] for i in ids]
                      and targets.dtype == torch.int64 and images.dtype == torch.float32
                      and tuple(images.shape) == (32, 3, 224, 224))
                images, targets = images.to(device), targets.to(device)
                check("finite_normalised_batch_" + phase + str(number), bool(torch.isfinite(images).all()))
                model.train(training)
                check("features_eval_" + phase + str(number), not any(m.training for m in model.network.features.modules()))
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    logits = model(images)
                    loss = criterion(logits, targets)
                    check("finite_loss_" + phase + str(number), logits.shape == (32, 8)
                          and bool(torch.isfinite(logits).all() & torch.isfinite(loss)))
                    loss.backward()
                    gradients = [p.grad for p in model.network.classifier.parameters()]
                    check("finite_nonzero_head_gradients_" + phase + str(number), all(g is not None and bool(torch.isfinite(g).all()) for g in gradients)
                          and any(bool(torch.any(g != 0)) for g in gradients))
                    optimizer.step()
                else:
                    with torch.inference_mode():
                        logits = model(images)
                        loss = criterion(logits, targets)
                    check("finite_loss_" + phase + str(number), logits.shape == (32, 8)
                          and bool(torch.isfinite(logits).all() & torch.isfinite(loss)))
                    validation_ids.extend(ids)
                    fixed = images
                torch.cuda.synchronize(device)
                previous = time.perf_counter() - batch_started
                result["batch_seconds"][phase].append(previous)
                result["completed_batches"][phase] += 1
                print(f"Disposable {phase} {number + 1}: {previous:.3f}s (loading included)", flush=True)
        check("validation_prefix_order", validation_ids == rows["validation"]["image_id"].tolist()[:len(validation_ids)])
        check("classifier_updated", any(not torch.equal(value.cpu(), head_before[name])
              for name, value in model.network.classifier.state_dict().items()))
        # Compare every feature parameter AND buffer, including all BatchNorm statistics/counters.
        check("frozen_parameters_and_bn_buffers_unchanged", all(torch.equal(value.cpu(), frozen_before[name])
              for name, value in model.network.features.state_dict().items())
              and all(p.grad is None for p in model.network.features.parameters()))
        result["checks"]["deterministic_evaluation"] = False
        result["checks"]["temporary_state_roundtrip"] = False
        if fixed is not None and room(90, previous * 2):
            model.eval()
            with torch.inference_mode():
                expected = model(fixed).clone()
                again = model(fixed)
            check("deterministic_evaluation", torch.equal(expected, again) and bool(torch.isfinite(expected).all()))
            if room(90, previous):
                with tempfile.NamedTemporaryFile(dir=output.parent, prefix="mobilenetv2_diagnostic_", suffix=".pt", delete=False) as stream:
                    temporary = Path(stream.name)
                torch.save(model.state_dict(), temporary)
                # No random fallback: this skeleton is immediately loaded strictly from the diagnostic state.
                reloaded = FrozenMobileNetV2(weights=None).to(device=device, dtype=torch.float32)
                reloaded.load_state_dict(torch.load(temporary, map_location=device, weights_only=True), strict=True)
                reloaded.eval()
                with torch.inference_mode():
                    actual = reloaded(fixed)
                torch.cuda.synchronize(device)
                check("temporary_state_roundtrip", torch.equal(expected, actual))
        check("verified_inputs_remain_unchanged", all(digest(setup / name) == value for name, value in actual_hashes.items())
              and digest(setup_report_path) == result["input_sha256"]["setup_report"]
              and digest(cache_report_path) == result["input_sha256"]["cache_report"])
        complete = result["completed_batches"] == {"warmup": 2, "training": 3, "validation": 2}
        result["status"] = "passed" if complete and all(result["checks"].values()) else "incomplete_budget_limited"
    except Exception as error:
        result["status"] = "failed"
        result["failures"].append(f"{type(error).__name__}: {error}; no random-weights fallback or automatic setup rerun")
        raise
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)  # Only the file created by this diagnostic.
        result["temporary_file_cleaned"] = temporary is None or not temporary.exists()
        model = optimizer = reloaded = fixed = None
        result["diagnostic_model_and_optimizer_discarded"] = True
        result["elapsed_diagnostic_seconds"] = time.perf_counter() - started if started is not None else None
        result["budget_overrun_seconds"] = max(0, result["elapsed_diagnostic_seconds"] - 90) if started is not None else None
        means = {name: statistics.mean(times) if times else None for name, times in result["batch_seconds"].items()}
        result["mean_batch_seconds"] = means
        seconds = (means["training"] * result["loader_batches"]["train"] + means["validation"] * result["loader_batches"]["validation"]
                   if means["training"] is not None and means["validation"] is not None else None)
        result["rough_epoch_projection"] = {"label": "Rough short-sample projection, NOT measured epoch time", "seconds": seconds,
                                           "minutes": seconds / 60 if seconds is not None else None}
        result["failed_checks"] = [name for name, passed in result["checks"].items() if not passed]
        if output.parent.is_dir():
            output.write_text(json.dumps(result, indent=2, allow_nan=False) + "\n", encoding="utf-8")
        print(json.dumps({key: result.get(key) for key in ["status", "parameters", "completed_batches", "mean_batch_seconds",
            "rough_epoch_projection", "failed_checks", "failures", "temporary_file_cleaned"]}, indent=2))
        print("Diagnostic report:", output, "-- full E1 training remains disabled")
    return result


mobilenetv2_setup_check = _verify_mobilenetv2_setup()

## 3. Guarded E1 training (disabled by default)

Paste the complete next code cell **after the already passed setup cell** in the current Kaggle notebook. Leave `RUN_E1_TRAINING=False` for a safe initial run; to deliberately start the real experiment, change only that flag to `True` in this new cell and run it once. Do not rerun diagnostics, restart, Run All or recopy images. The successful live diagnostic must equal its on-disk JSON and all input/weight hashes must match. Missing/changed state or an existing E1 directory stops execution.

A fresh seed42 pretrained model trains only its classifier for at most 10 epochs, with sample-weighted metrics, early stopping (3), and ReduceLROnPlateau (factor0.5, patience2, absolute threshold0). Frozen features and BatchNorm buffers are checked each epoch. Outputs go exclusively to `/kaggle/working/E1_mobilenetv2_frozen_seed42`: configuration, per-epoch history and the best checkpoint. Interrupted directories are preserved; no automatic overwrite/resume and no exact-resume guarantee. The 4.24-minute setup projection is not measured epoch time; use actual completed epoch timings.


In [ ]:
# E1 FULL TRAINING: change only this flag when ready to start the real run.
RUN_E1_TRAINING = False
E1_RUN_COMPLETED = False
import hashlib
import json
import platform
import sys
import time
from pathlib import Path
from importlib.metadata import version
from urllib.parse import urlparse
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import MobileNet_V2_Weights


def _e1_require_diagnostic(live, saved):
    if live != saved:
        raise RuntimeError("Live MobileNetV2 diagnostic differs from the on-disk report")
    if (saved.get("status") != "passed" or saved.get("failures") or saved.get("failed_checks")
            or not saved.get("checks") or not all(value is True for value in saved["checks"].values())
            or not saved.get("temporary_file_cleaned") or not saved.get("diagnostic_model_and_optimizer_discarded")
            or saved.get("parameters") != {"total": 2234120, "trainable": 10248}
            or saved.get("device", {}).get("selected") != "cuda:0"):
        raise RuntimeError("A complete successful hosted MobileNetV2 diagnostic is required")


def _e1_check_frozen(model, before):
    features = model.network.features
    if (any(module.training for module in features.modules())
            or any(p.requires_grad or p.grad is not None for p in features.parameters())
            or set(features.state_dict()) != set(before)
            or any(not torch.equal(value.detach().cpu(), before[name])
                   for name, value in features.state_dict().items())):
        raise RuntimeError("Frozen feature parameters or BatchNorm buffers/mode changed")


def _e1_epoch(model, loader, criterion, device, optimizer=None):
    """Sample-weighted metrics; saved IDs ensure full coverage, including the tail."""
    training = optimizer is not None
    model.train(training)  # Wrapper keeps features/BatchNorm in eval mode.
    if any(module.training for module in model.network.features.modules()):
        raise RuntimeError("Frozen feature extractor entered training mode")
    total_loss = 0.0
    correct = samples = 0
    seen = []
    expected_ids = loader.dataset.base.rows["image_id"].tolist()
    with (torch.enable_grad() if training else torch.inference_mode()), torch.autocast(device_type=device.type, enabled=False):
        for images, targets, image_ids in loader:
            if images.dtype != torch.float32 or targets.dtype != torch.int64:
                raise ValueError("Expected float32 tensors and int64 labels")
            images, targets = images.to(device), targets.to(device)
            if not bool(torch.isfinite(images).all()) or not bool(((targets >= 0) & (targets < 8)).all()):
                raise ValueError("Nonfinite inputs or labels outside 0-7")
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            if logits.shape != (len(image_ids), 8) or not bool(torch.isfinite(logits).all() & torch.isfinite(loss)):
                raise FloatingPointError("Invalid logits or nonfinite cross-entropy")
            if training:
                loss.backward()
                gradients = [p.grad for p in model.network.classifier.parameters()]
                if not gradients or any(g is None or not bool(torch.isfinite(g).all()) for g in gradients):
                    raise FloatingPointError("Missing/nonfinite classifier gradients")
                optimizer.step()
            size = len(image_ids)
            total_loss += float(loss.detach()) * size
            correct += int((logits.detach().argmax(dim=1) == targets).sum())
            samples += size
            seen.extend(image_ids)
    if (samples != len(loader.dataset) or samples == 0 or len(set(seen)) != samples
            or set(seen) != set(expected_ids) or (not training and seen != expected_ids)):
        raise ValueError("Epoch sample totals, ID coverage or validation order changed")
    return {"loss": total_loss / samples, "accuracy": correct / samples, "samples": samples}


def _run_e1_training():
    required = ["FrozenMobileNetV2", "ImageNetNormalizedDataset", "SETUP_ROOT", "TRAIN_CACHE_DIR", "mobilenetv2_setup_check"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Missing current-session state: " + ", ".join(missing) + "; nothing rerun automatically")
    working = Path("/kaggle/working")
    run = working / "E1_mobilenetv2_frozen_seed42"
    if run.exists() or run.is_symlink():
        raise FileExistsError(f"E1 directory already exists; no overwrite/resume: {run}")
    report_path = working / "mobilenetv2_setup_check.json"
    if not report_path.is_file():
        raise FileNotFoundError(f"Required hosted diagnostic missing: {report_path}")
    diagnostic = json.loads(report_path.read_text())
    _e1_require_diagnostic(mobilenetv2_setup_check, diagnostic)

    def digest(path):
        with Path(path).open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    setup, cache = Path(SETUP_ROOT).resolve(), Path(TRAIN_CACHE_DIR).resolve()
    pinned = "14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
    if digest(setup / "transfer_manifest.json") != pinned:
        raise ValueError("Pinned transfer manifest changed")
    transfer = json.loads((setup / "transfer_manifest.json").read_text())
    expected_hashes = {**transfer["sha256"], "transfer_manifest.json": pinned}
    inputs = {name: setup / name for name in expected_hashes}
    inputs.update(setup_report=working / "gpu_setup_check.json", cache_report=working / "training_image_cache_check.json")
    hashes = {name: digest(path) for name, path in inputs.items()}
    if hashes != diagnostic["input_sha256"] or any(hashes[name] != value for name, value in expected_hashes.items()):
        raise ValueError("Verified source/split/mapping/setup/cache-report hashes changed")
    cache_report = json.loads(inputs["cache_report"].read_text())
    if (not cache.is_dir() or cache != (working / "enge707_train_cache").resolve()
            or cache_report.get("status") != "passed" or cache_report.get("verified_files") != 63660
            or cache_report.get("failures") or cache_report.get("hash_mismatches")
            or not cache_report.get("checks") or not all(cache_report["checks"].values())
            or Path(cache_report["cache_directory"]).resolve() != cache):
        raise RuntimeError("Verified working-image cache is unavailable")
    module = sys.modules.get("src.data_pipeline")
    if module is None or Path(module.__file__).resolve() != setup / "src/data_pipeline.py":
        raise RuntimeError("Verified Dataset module is missing or loaded from another location")
    from src.data_pipeline import FacialFeatureDataset, seed_everything
    split = pd.read_csv(setup / "data/processed/split_manifest.csv", dtype=str, keep_default_na=False, na_filter=False)
    mapping = json.loads((setup / "data/processed/class_mapping.json").read_text())
    names = sorted(mapping["class_to_index"])
    counts = {"train": 50927, "validation": 12733}
    if (mapping != diagnostic["class_mapping"] or mapping != transfer["classes"] or len(names) != 8
            or mapping["class_to_index"] != dict(zip(names, range(8)))
            or any(mapping["index_to_class"][str(i)] != name for i, name in enumerate(names))
            or len(split) != 63660 or not split["image_id"].is_unique
            or not split["image_id"].str.fullmatch(r"[0-9]{5}").all()
            or split["split"].value_counts().to_dict() != counts or diagnostic["dataset_lengths"] != counts):
        raise ValueError("Saved split or exact class encoding changed")
    rows = {name: split.loc[split["split"].eq(name)].reset_index(drop=True) for name in counts}
    if (any(set(rows["train"][key]) & set(rows["validation"][key]) for key in ["image_id", "byte_sha256", "pixel_sha256"])
            or any(set(frame["class"]) != set(names) for frame in rows.values())):
        raise ValueError("Split disjointness or class coverage changed")
    weights = MobileNet_V2_Weights.IMAGENET1K_V2
    provenance = diagnostic["weights"]
    weight_file = Path(torch.hub.get_dir()) / "checkpoints" / Path(urlparse(weights.url).path).name
    if (provenance["identifier"] != "MobileNet_V2_Weights.IMAGENET1K_V2"
            or provenance["url"] != weights.url or Path(provenance["cache_path"]).resolve() != weight_file.resolve()
            or not weight_file.is_file() or digest(weight_file) != provenance["sha256"]
            or not provenance["sha256"].startswith("7ebf99e0")):
        raise RuntimeError("Previously verified pretrained-weight file is missing/changed; no download attempted")
    if not torch.cuda.is_available():
        raise RuntimeError("Required cuda:0 is unavailable")
    device = torch.device("cuda:0")
    seed_everything(42)
    bases = {name: FacialFeatureDataset(setup, name, training_image_directory=cache) for name in counts}
    if (any(len(base) != counts[name] or not base.rows.equals(rows[name][["image_id", "class", "image_path"]])
            or base.class_to_index != mapping["class_to_index"] for name, base in bases.items())
            or bases["validation"].augmentation is not None
            or repr(bases["train"].augmentation) != diagnostic["training_augmentation"]):
        raise ValueError("Fresh datasets differ from verified membership/order/augmentation")
    datasets = {name: ImageNetNormalizedDataset(base) for name, base in bases.items()}
    for dataset in datasets.values():
        if (not torch.equal(dataset.mean.flatten(), torch.tensor([0.485, 0.456, 0.406]))
                or not torch.equal(dataset.std.flatten(), torch.tensor([0.229, 0.224, 0.225]))):
            raise ValueError("ImageNet normalisation changed")
    loaders = {"train": DataLoader(datasets["train"], batch_size=32, shuffle=True,
        generator=torch.Generator().manual_seed(42), num_workers=0, drop_last=False),
        "validation": DataLoader(datasets["validation"], batch_size=32, shuffle=False, num_workers=0, drop_last=False)}
    # Exclusive reservation also protects an interrupted run; never automatically resume.
    run.mkdir(exist_ok=False)
    model = FrozenMobileNetV2(weights=weights).to(device=device, dtype=torch.float32)
    parameters = {"total": sum(p.numel() for p in model.parameters()),
                  "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
    if (parameters != diagnostic["parameters"] or any(p.requires_grad for p in model.network.features.parameters())
            or not all(p.requires_grad for p in model.network.classifier.parameters())):
        raise ValueError("Model architecture/trainability changed")
    frozen = {name: value.detach().cpu().clone() for name, value in model.network.features.state_dict().items()}
    _e1_check_frozen(model, frozen)
    optimizer = torch.optim.Adam(model.network.classifier.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
        patience=2, threshold=0, threshold_mode="abs")
    configuration = {"run_name": run.name, "seed": 42, "max_epochs": 10, "early_stopping_patience": 3,
        "optimizer": "Adam", "optimizer_parameters": "classifier only", "learning_rate": 0.001,
        "loss": "unweighted CrossEntropyLoss on raw logits", "best_checkpoint_rule": "lowest validation loss",
        "scheduler": {"name": "ReduceLROnPlateau", "mode": "min", "factor": 0.5, "patience": 2,
                      "threshold": 0, "threshold_mode": "abs"},
        "batch_size": 32, "num_workers": 0, "drop_last": False, "train_shuffle": True, "validation_shuffle": False,
        "device": "cuda:0", "dtype": "float32", "amp": False, "multi_gpu": False,
        "architecture": "FrozenMobileNetV2; Dropout(0.3), Linear(last_channel,8), raw logits", "parameters": parameters,
        "weights": provenance, "preprocessing": diagnostic["preprocessing"],
        "training_augmentation": repr(bases["train"].augmentation), "validation_augmentation": None,
        "trainability_policy": "Only classifier trains; features and all BatchNorm statistics stay frozen/eval",
        "class_mapping": mapping, "counts": counts, "training_image_directory": str(cache),
        "split_sha256": hashes["data/processed/split_manifest.csv"], "input_sha256": hashes,
        "diagnostic_sha256": digest(report_path),
        "runtime": {"python": platform.python_version(), "torch": str(torch.__version__),
                    "torchvision": torchvision.__version__, "pandas": pd.__version__,
                    "numpy": version("numpy"), "Pillow": version("Pillow"), "cuda_build": torch.version.cuda},
        "gpu": {"selected": str(device), "name": torch.cuda.get_device_name(0)},
        "reproducibility": "Fresh seeded pretrained model; no cross-version/GPU bitwise or exact-resume guarantee"}
    with (run / "configuration.json").open("x", encoding="utf-8") as stream:
        json.dump(configuration, stream, indent=2, allow_nan=False)
        stream.write("\n")
    history = []
    best_loss, best_epoch, without_improvement = float("inf"), None, 0
    torch.cuda.synchronize(device)
    started = time.perf_counter()
    for epoch in range(1, 11):
        torch.cuda.synchronize(device)
        epoch_started = time.perf_counter()
        learning_rate = float(optimizer.param_groups[0]["lr"])
        training = _e1_epoch(model, loaders["train"], criterion, device, optimizer)
        validation = _e1_epoch(model, loaders["validation"], criterion, device)
        if training["samples"] != 50927 or validation["samples"] != 12733:
            raise ValueError("Saved split sample totals not met")
        _e1_check_frozen(model, frozen)
        if validation["loss"] < best_loss:
            best_loss, best_epoch, without_improvement = validation["loss"], epoch, 0
            torch.save({"model_state_dict": model.state_dict(), "class_mapping": mapping,
                "configuration": configuration, "split_checksum": configuration["split_sha256"],
                "best_epoch": best_epoch, "best_validation_loss": best_loss}, run / "best_checkpoint.pt")
        else:
            without_improvement += 1
        scheduler.step(validation["loss"])
        torch.cuda.synchronize(device)
        row = {"epoch": epoch, "training_loss": training["loss"], "validation_loss": validation["loss"],
            "training_accuracy": training["accuracy"], "validation_accuracy": validation["accuracy"],
            "training_samples": training["samples"], "validation_samples": validation["samples"],
            "learning_rate": learning_rate, "next_learning_rate": float(optimizer.param_groups[0]["lr"]),
            "epoch_elapsed_seconds": time.perf_counter() - epoch_started, "elapsed_seconds": time.perf_counter() - started}
        history.append(row)
        pd.DataFrame(history).to_csv(run / "history.csv", index=False)
        print(row, flush=True)
        if without_improvement >= 3:
            print("Early stopping: 3 consecutive epochs without lower validation loss", flush=True)
            break
    _e1_check_frozen(model, frozen)
    if (any(digest(path) != hashes[name] for name, path in inputs.items())
            or digest(weight_file) != provenance["sha256"] or digest(report_path) != configuration["diagnostic_sha256"]):
        raise ValueError("Verified inputs changed during E1")
    torch.cuda.synchronize(device)
    result = {"status": "completed", "directory": str(run), "completed_epochs": len(history),
        "best_epoch": best_epoch, "best_validation_loss": best_loss, "elapsed_seconds": time.perf_counter() - started,
        "frozen_parameters_and_bn_buffers_unchanged": True, "history_sha256": digest(run / "history.csv")}
    print(json.dumps(result, indent=2), flush=True)
    return result


if RUN_E1_TRAINING:
    e1_training_result = _run_e1_training()
    E1_RUN_COMPLETED = True  # Set only after all epochs/checks finish successfully.
else:
    print("RUN_E1_TRAINING=False: no E1 directory, training, history or checkpoints created.")

## 4. Guarded learning curves

Paste this as a **separate cell immediately after the training cell**. Run it only after the deliberately enabled E1 run completes successfully. With the default flag False, it safely skips all plotting/output. It verifies actual saved history and refuses to overwrite an existing curve image. No diagnostic results are plotted.


In [ ]:
# E1 LEARNING CURVES: actual completed experiment history only.
if RUN_E1_TRAINING:
    if (not globals().get("E1_RUN_COMPLETED", False)
            or globals().get("e1_training_result", {}).get("status") != "completed"):
        raise RuntimeError("Complete the guarded E1 training cell successfully before plotting")
    import matplotlib.pyplot as plt
    run = Path(e1_training_result["directory"])
    history_path = run / "history.csv"
    with history_path.open("rb") as stream:
        actual_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    if (run != Path("/kaggle/working/E1_mobilenetv2_frozen_seed42")
            or actual_hash != e1_training_result["history_sha256"]):
        raise ValueError("Completed E1 history path or checksum changed")
    frame = pd.read_csv(history_path, float_precision="round_trip")
    if (len(frame) != e1_training_result["completed_epochs"]
            or frame["epoch"].tolist() != list(range(1, len(frame) + 1))):
        raise ValueError("Unexpected completed E1 history")
    figure_path = run / "loss_accuracy_curves.png"
    if figure_path.exists() or figure_path.is_symlink():
        raise FileExistsError(f"Existing E1 curves left unchanged: {figure_path}")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for partition, label in [("training", "Training"), ("validation", "Validation")]:
        axes[0].plot(frame["epoch"], frame[partition + "_loss"], label=label)
        axes[1].plot(frame["epoch"], frame[partition + "_accuracy"], label=label)
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss", title="E1 loss")
    axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="E1 accuracy", ylim=(0, 1))
    for axis in axes:
        axis.legend(frameon=False)
        axis.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    with figure_path.open("xb") as stream:
        fig.savefig(stream, format="png", dpi=300, facecolor="white")
    plt.show()
    plt.close(fig)
    print("Saved actual E1 learning curves:", figure_path)
else:
    print("RUN_E1_TRAINING=False: learning-curve output skipped.")

In [ ]:
# 5. E1 EVALUATION ONLY: restore epoch 10; no setup, downloads or training.
RUN_E1_TRAINING = False
RUN_FULL_TRAINING = False
try:
    import hashlib
    import json
    import math
    import platform
    import sys
    from pathlib import Path
    from importlib.metadata import version
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    import torchvision
    from torch.utils.data import DataLoader
    from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
except ImportError as error:
    raise RuntimeError("Required evaluation dependency is unavailable; nothing installed") from error


def _e1_validation_statistics(true, predicted, loss_sum, names):
    """Use explicit class order; undefined precision/F1 is reported as zero."""
    precision, recall, f1, support = precision_recall_fscore_support(
        true, predicted, labels=list(range(8)), zero_division=0)
    matrix = confusion_matrix(true, predicted, labels=list(range(8)))
    per_class = pd.DataFrame({"class_index": list(range(8)), "class": names,
        "precision": precision, "recall": recall, "f1": f1, "support": support})
    scores = {"samples": len(true), "cross_entropy": loss_sum / len(true),
        "accuracy": float(accuracy_score(true, predicted)), "macro_f1": float(f1.mean()),
        "weighted_f1": float(np.average(f1, weights=support))}
    return scores, per_class, matrix


def _evaluate_saved_e1_validation():
    required = ["FrozenMobileNetV2", "ImageNetNormalizedDataset", "SETUP_ROOT", "TRAIN_CACHE_DIR", "mobilenetv2_setup_check"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Missing verified session state: " + ", ".join(missing) + "; no setup/copy/training rerun")
    run = Path("/kaggle/working/E1_mobilenetv2_frozen_seed42")
    destination = run / "evaluation"
    if destination.exists() or destination.is_symlink():
        raise FileExistsError(f"Existing evaluation directory left unchanged: {destination}")
    setup, cache = Path(SETUP_ROOT).resolve(), Path(TRAIN_CACHE_DIR).resolve()
    paths = {"checkpoint": run / "best_checkpoint.pt", "configuration": run / "configuration.json",
        "history": run / "history.csv", "diagnostic": run.parent / "mobilenetv2_setup_check.json"}
    for path in [*paths.values(), setup / "transfer_manifest.json"]:
        if not path.is_file():
            raise FileNotFoundError(f"Required existing file unavailable: {path}")

    def digest(path):
        with Path(path).open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    history_hash = "b0bbd7134ca16a70fbb480b8895b5debf97007621de16aaa52ad88ec1a36b3ea"
    pinned = "14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
    if digest(paths["history"]) != history_hash or digest(setup / "transfer_manifest.json") != pinned:
        raise ValueError("Completed history or pinned transfer-manifest checksum changed")
    transfer = json.loads((setup / "transfer_manifest.json").read_text())
    expected_hashes = {**transfer["sha256"], "transfer_manifest.json": pinned}
    paths.update({name: setup / name for name in expected_hashes})
    paths.update(setup_report=run.parent / "gpu_setup_check.json", cache_report=run.parent / "training_image_cache_check.json")
    before = {name: digest(path) for name, path in paths.items()}
    config = json.loads(paths["configuration"].read_text())
    diagnostic = json.loads(paths["diagnostic"].read_text())
    if (diagnostic != mobilenetv2_setup_check or diagnostic.get("status") != "passed"
            or diagnostic.get("failures") or diagnostic.get("failed_checks")
            or not diagnostic.get("checks") or not all(v is True for v in diagnostic["checks"].values())
            or before["diagnostic"] != config["diagnostic_sha256"]):
        raise ValueError("Successful hosted diagnostic is missing, changed or inconsistent")
    if (config["input_sha256"] != diagnostic["input_sha256"]
            or any(before[name] != expected for name, expected in config["input_sha256"].items())
            or any(before[name] != expected for name, expected in expected_hashes.items())):
        raise ValueError("Recorded source/split/mapping/setup/cache hashes changed")
    module = sys.modules.get("src.data_pipeline")
    if module is None or Path(module.__file__).resolve() != setup / "src/data_pipeline.py":
        raise RuntimeError("Existing verified Dataset module unavailable or from another path")
    from src.data_pipeline import FacialFeatureDataset
    history = pd.read_csv(paths["history"], float_precision="round_trip")
    checkpoint = torch.load(paths["checkpoint"], weights_only=True, map_location="cpu")
    if (config["run_name"] != run.name or config["seed"] != 42 or config["max_epochs"] != 10
            or checkpoint["configuration"] != config or checkpoint["best_epoch"] != 10
            or history["epoch"].tolist() != list(range(1, 11))
            or int(history.loc[history["validation_loss"].idxmin(), "epoch"]) != 10
            or not history["training_samples"].eq(50927).all()
            or not history["validation_samples"].eq(12733).all()):
        raise ValueError("Configuration/checkpoint/history does not describe completed E1 epoch 10")
    epoch10 = history.iloc[-1]
    if not math.isclose(checkpoint["best_validation_loss"], float(epoch10["validation_loss"]), rel_tol=0, abs_tol=1e-12):
        raise ValueError("Checkpoint best loss differs from completed history")
    mapping = json.loads((setup / "data/processed/class_mapping.json").read_text())
    names = [mapping["index_to_class"][str(i)] for i in range(8)]
    if (names != sorted(names) or len(set(names)) != 8
            or mapping["class_to_index"] != dict(zip(names, range(8)))
            or mapping != config["class_mapping"] or mapping != checkpoint["class_mapping"]
            or mapping != diagnostic["class_mapping"] or mapping != transfer["classes"]):
        raise ValueError("Exact saved class mapping changed")
    split_hash = before["data/processed/split_manifest.csv"]
    if config["split_sha256"] != split_hash or checkpoint["split_checksum"] != split_hash:
        raise ValueError("Checkpoint/configuration split checksum mismatch")
    cache_report = json.loads(paths["cache_report"].read_text())
    if (not cache.is_dir() or cache != (run.parent / "enge707_train_cache").resolve()
            or cache_report.get("status") != "passed" or cache_report.get("verified_files") != 63660
            or cache_report.get("failures") or cache_report.get("hash_mismatches")
            or Path(cache_report["cache_directory"]).resolve() != cache):
        raise RuntimeError("Verified working-image cache unavailable; nothing recopied")
    split = pd.read_csv(setup / "data/processed/split_manifest.csv", dtype=str, keep_default_na=False, na_filter=False)
    if (len(split) != 63660 or not split["image_id"].is_unique
            or not split["image_id"].str.fullmatch(r"[0-9]{5}").all()
            or split["split"].value_counts().to_dict() != {"train": 50927, "validation": 12733}):
        raise ValueError("Saved split coverage or original IDs changed")
    expected = split.loc[split["split"].eq("validation")].reset_index(drop=True)
    base = FacialFeatureDataset(setup, "validation", training_image_directory=cache)
    if (len(base) != 12733 or base.augmentation is not None
            or base.class_to_index != mapping["class_to_index"]
            or not base.rows.equals(expected[["image_id", "class", "image_path"]])):
        raise ValueError("Validation membership/order/labels/augmentation changed")
    dataset = ImageNetNormalizedDataset(base)
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]
    if (not torch.equal(dataset.mean, mean) or not torch.equal(dataset.std, std)
            or config["preprocessing"] != diagnostic["preprocessing"]
            or config["preprocessing"]["normalisation_applications"] != 1):
        raise ValueError("Verified ImageNet normalisation changed")
    raw, label, image_id = base[0]
    first, label1, id1 = dataset[0]
    repeated, label2, id2 = dataset[0]
    if (not torch.equal(first, (raw - mean) / std) or not torch.equal(first, repeated)
            or label != label1 or label != label2 or image_id != id1 or image_id != id2):
        raise ValueError("Validation preprocessing is not deterministic or normalised exactly once")
    expected_ids = expected["image_id"].tolist()
    expected_targets = expected["class"].map(mapping["class_to_index"]).tolist()
    expected_support = [int(expected["class"].eq(name).sum()) for name in names]
    if not all(expected_support) or not torch.cuda.is_available():
        raise RuntimeError("All eight validation classes and cuda:0 are required")
    device = torch.device("cuda:0")
    # Restore ALL features and BatchNorm buffers from the checkpoint; no weight download.
    model = FrozenMobileNetV2(weights=None).to(dtype=torch.float32)
    parameters = {"total": sum(p.numel() for p in model.parameters()),
                  "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
    if parameters != config["parameters"] or parameters != {"total": 2234120, "trainable": 10248}:
        raise ValueError("E1 architecture/parameter counts changed")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(device).eval()
    buffers_before = {name: value.detach().cpu().clone() for name, value in model.network.features.named_buffers()}
    loader = DataLoader(dataset, batch_size=32, num_workers=0, drop_last=False, shuffle=False)
    criterion = torch.nn.CrossEntropyLoss()
    destination.mkdir(exist_ok=False)  # Reserve exclusively before evaluating.
    ids, true, predicted, confidence = [], [], [], []
    loss_sum = 0.0
    with torch.inference_mode(), torch.autocast(device_type="cuda", enabled=False):
        for batch_number, (images, targets, image_ids) in enumerate(loader, 1):
            start, end = len(ids), len(ids) + len(image_ids)
            if list(image_ids) != expected_ids[start:end] or targets.tolist() != expected_targets[start:end]:
                raise ValueError("Validation IDs/labels differ from exact saved order")
            if (images.dtype != torch.float32 or targets.dtype != torch.int64
                    or tuple(images.shape) != (len(image_ids), 3, 224, 224)):
                raise ValueError("Unexpected input shape or precision")
            images, targets = images.to(device), targets.to(device)
            if not bool(torch.isfinite(images).all()):
                raise FloatingPointError("Nonfinite normalised inputs")
            logits = model(images)
            loss = criterion(logits, targets)
            if (logits.shape != (len(image_ids), 8) or logits.dtype != torch.float32
                    or not bool(torch.isfinite(logits).all() & torch.isfinite(loss))):
                raise FloatingPointError("Invalid logits or loss")
            maximum, indices = logits.softmax(dim=1).max(dim=1)
            loss_sum += float(loss) * len(image_ids)  # Weight the final partial batch correctly.
            ids.extend(image_ids)
            true.extend(targets.cpu().tolist())
            predicted.extend(indices.cpu().tolist())
            confidence.extend(maximum.cpu().tolist())
            if batch_number % 100 == 0 or batch_number == len(loader):
                print(f"E1 evaluation only: {len(ids):,}/12,733 validation images", flush=True)
    scores, per_class, matrix = _e1_validation_statistics(true, predicted, loss_sum, names)
    normalised = matrix / matrix.sum(axis=1, keepdims=True)
    checks = {
        "exact_unique_id_coverage_and_order": ids == expected_ids and len(set(ids)) == 12733,
        "true_labels_match": true == expected_targets,
        "class_supports_match": per_class["support"].tolist() == expected_support,
        "prediction_indices_in_range": all(0 <= value < 8 for value in predicted),
        "finite_metrics_and_confidences": bool(np.isfinite(list(scores.values())).all()
            and np.isfinite(per_class[["precision", "recall", "f1"]].to_numpy()).all() and np.isfinite(confidence).all()),
        "confidence_in_range": all(0 <= value <= 1 for value in confidence),
        "confusion_total_12733": int(matrix.sum()) == 12733,
        "confusion_supports_match": matrix.sum(axis=1).tolist() == expected_support,
        "normalised_rows_sum_to_one": bool(np.allclose(normalised.sum(axis=1), 1)),
        "eval_no_gradients": not model.training and all(p.grad is None for p in model.parameters()),
        "batchnorm_buffers_unchanged": all(torch.equal(value.cpu(), buffers_before[name])
            for name, value in model.network.features.named_buffers()),
        "original_inputs_unchanged": all(digest(path) == before[name] for name, path in paths.items()),
    }
    if not all(checks.values()):
        raise ValueError("Evaluation integrity failed: " + str([name for name, passed in checks.items() if not passed]))
    comparison = {"epoch": 10, "history_accuracy": float(epoch10["validation_accuracy"]),
        "history_loss": float(epoch10["validation_loss"]),
        "accuracy_difference": scores["accuracy"] - float(epoch10["validation_accuracy"]),
        "loss_difference": scores["cross_entropy"] - float(epoch10["validation_loss"]),
        "accuracy_matches_exactly": scores["accuracy"] == float(epoch10["validation_accuracy"]),
        "loss_absolute_tolerance": 1e-6, "loss_matches_within_tolerance": math.isclose(
            scores["cross_entropy"], float(epoch10["validation_loss"]), rel_tol=0, abs_tol=1e-6)}
    discrepancy = not (comparison["accuracy_matches_exactly"] and comparison["loss_matches_within_tolerance"])
    warnings = ["Restored metrics differ from epoch 10; actual results were retained."] if discrepancy else []
    predictions = pd.DataFrame({"image_id": ids, "true_class": [names[i] for i in true],
        "predicted_class": [names[i] for i in predicted], "max_softmax_confidence": confidence})
    with (destination / "validation_predictions.csv").open("x", newline="", encoding="utf-8") as stream:
        predictions.to_csv(stream, index=False)
    with (destination / "classification_report.csv").open("x", newline="", encoding="utf-8") as stream:
        per_class.to_csv(stream, index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), layout="constrained")
    for ax, values, title, fractional in zip(axes, [matrix, normalised],
            ["Raw counts", "Row-normalised (true class)"], [False, True]):
        image = ax.imshow(values, cmap="Blues", vmin=0, vmax=1 if fractional else matrix.max())
        ax.set(xticks=range(8), yticks=range(8), xticklabels=names, yticklabels=names,
               xlabel="Predicted class", ylabel="True class", title=title)
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
        for row in range(8):
            for column in range(8):
                value = values[row, column]
                ax.text(column, row, f"{value:.2f}" if fractional else str(value),
                        ha="center", va="center", fontsize=8.5,
                        color="white" if value > (0.5 if fractional else matrix.max() / 2) else "#25323A")
        fig.colorbar(image, ax=ax, shrink=0.8)
    fig.suptitle("E1 validation - saved checkpoint, epoch 10")
    with (destination / "confusion_matrix.png").open("xb") as stream:
        fig.savefig(stream, format="png", dpi=300, facecolor="white")
    plt.show()
    plt.close(fig)
    files = [str(destination / name) for name in ["validation_metrics.json", "classification_report.csv",
             "validation_predictions.csv", "confusion_matrix.png"]]
    metrics = {"status": "passed_with_history_discrepancy" if discrepancy else "passed",
        "evaluation_only": True, "run_name": run.name, "best_epoch": 10, **scores,
        "device": str(device), "gpu": torch.cuda.get_device_name(0), "dtype": "float32", "amp": False,
        "batch_size": 32, "num_workers": 0, "drop_last": False, "shuffle": False, "augmentation": False,
        "parameters": parameters, "weights_provenance": config["weights"], "preprocessing": config["preprocessing"],
        "class_mapping": mapping, "per_class": per_class.to_dict(orient="records"),
        "confusion_matrix": matrix.tolist(), "row_normalised_confusion_matrix": normalised.tolist(),
        "input_sha256": before, "checks": checks, "failed_checks": [], "history_comparison": comparison,
        "warnings": warnings, "outputs": files,
        "runtime": {"python": platform.python_version(), "torch": str(torch.__version__),
                    "torchvision": torchvision.__version__, "numpy": np.__version__, "pandas": pd.__version__,
                    "Pillow": version("Pillow"), "cuda_build": torch.version.cuda},
        "limitations": ["No cross-version/GPU bitwise reproducibility claim.", "Undefined precision/F1 uses zero_division=0."]}
    with (destination / "validation_metrics.json").open("x", encoding="utf-8") as stream:
        json.dump(metrics, stream, indent=2, allow_nan=False)
        stream.write("\n")
    print(json.dumps({"status": metrics["status"], **scores, "history_comparison": comparison,
                      "warnings": warnings, "outputs": files}, indent=2))
    return metrics


e1_validation_metrics = _evaluate_saved_e1_validation()

In [ ]:
# 6. GUARDED E1 TEST PREDICTION: enable deliberately; never upload automatically.
RUN_TEST_PREDICTION = False
RUN_E1_TRAINING = False
RUN_FULL_TRAINING = False
import hashlib
import json
import math
import os
import platform
import re
import statistics
import sys
import time
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader


def _e1_discover_test_input(root):
    """Enumerate input filenames once; no image decoding or image-content hashing."""
    root = Path(root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Supplied dataset directory unavailable: {root}")
    samples, image_directories = [], {}
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
    for directory, folders, files in os.walk(root, followlinks=False):
        directory = Path(directory)
        if any((directory / name).is_symlink() for name in folders):
            raise ValueError(f"Linked dataset directory is ambiguous: {directory}")
        for name in files:
            path = directory / name
            if name == "sample_submission.csv":
                if path.is_symlink():
                    raise ValueError("Sample submission must not be a link")
                samples.append(path)
            if "test" in directory.relative_to(root).parts and path.suffix.lower() in suffixes:
                if path.is_symlink():
                    raise ValueError(f"Linked test image is not allowed: {path}")
                image_directories.setdefault(directory, []).append(name)
    if len(samples) != 1 or len(image_directories) != 1:
        raise ValueError(f"Expected one sample CSV and one test-image directory; found {len(samples)} and {len(image_directories)}")
    directory, filenames = next(iter(image_directories.items()))
    return samples[0], directory.resolve(), filenames


def _e1_test_lookup(sample, filenames, expected_count=29443):
    if list(sample.columns) != ["image_id", "class"] or len(sample) != expected_count:
        raise ValueError("Expected sample columns image_id,class and exactly 29,443 rows")
    ids = sample["image_id"].tolist()
    if (any(not isinstance(i, str) or not re.fullmatch(r"[1-9][0-9]*", i) for i in ids)
            or len(set(ids)) != expected_count):
        raise ValueError("Expected unique nonempty original unpadded decimal string IDs")
    lookup = [image_id.zfill(5) + ".jpg" for image_id in ids]  # File lookup ONLY.
    if (len(set(lookup)) != expected_count or len(filenames) != expected_count
            or len(set(filenames)) != expected_count or set(lookup) != set(filenames)
            or any(not re.fullmatch(r"[0-9]{5}\.jpg", name) for name in filenames)):
        raise ValueError("Test filenames have missing, ambiguous, reused or unused images")
    return ids, lookup  # Never replace original sample IDs with padded IDs.


def _e1_test_tensor(image, preprocess, mean, std):
    tensor = preprocess(image)  # Existing RGB, bilinear resize/pad, float32 /255.
    if tensor.dtype != torch.float32 or tensor.shape != (3, 224, 224):
        raise ValueError("Unexpected reference-preprocessing output")
    return (tensor - mean) / std  # Same E1 normalisation, once; no crop/augmentation.


class E1TestDataset(Dataset):
    def __init__(self, directory, ids, lookup, preprocess, mean, std):
        self.directory = Path(directory).resolve()
        self.ids, self.lookup = list(ids), list(lookup)
        self.preprocess, self.mean, self.std = preprocess, mean, std

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        path = self.directory / self.lookup[index]
        if path.is_symlink() or path.resolve().parent != self.directory:
            raise ValueError(f"Test path escapes its directory: {path}")
        with Image.open(path) as image:
            image.load()  # Decode errors propagate; never skip an image.
            tensor = _e1_test_tensor(image, self.preprocess, self.mean, self.std)
        return tensor, self.ids[index]


def _e1_submission_frame(ids, indices, index_to_class, expected_count=29443):
    if (len(ids) != expected_count or len(set(ids)) != expected_count or len(indices) != expected_count
            or any(not isinstance(i, str) or not i.strip() for i in ids)
            or any(not isinstance(i, (int, np.integer)) or not 0 <= i < 8 for i in indices)):
        raise ValueError("Incomplete IDs or invalid prediction indices")
    classes = [index_to_class[str(index)] for index in indices]
    if not set(classes) <= set(index_to_class.values()):
        raise ValueError("Prediction outside official class encoding")
    return pd.DataFrame({"image_id": list(ids), "class": classes})


def _predict_e1_test():
    working = Path("/kaggle/working")
    submission_path, check_path = working / "submission_E1.csv", working / "submission_E1_check.json"
    for path in [submission_path, check_path]:
        if path.exists() or path.is_symlink():
            raise FileExistsError(f"Existing output left unchanged: {path}")
    missing = [name for name in ["FrozenMobileNetV2", "ImageNetNormalizedDataset", "SETUP_ROOT"] if name not in globals()]
    if missing:
        raise RuntimeError("Missing verified session definitions: " + ", ".join(missing) + "; no setup rerun")
    run = working / "E1_mobilenetv2_frozen_seed42"
    setup = Path(SETUP_ROOT).resolve()
    evaluation_path = run / "evaluation/validation_metrics.json"
    if not evaluation_path.is_file():
        raise FileNotFoundError(f"Successful E1 evaluation is required: {evaluation_path}")
    evaluation = json.loads(evaluation_path.read_text())
    if (evaluation.get("status") != "passed" or evaluation.get("samples") != 12733
            or evaluation.get("run_name") != run.name or evaluation.get("best_epoch") != 10
            or evaluation.get("evaluation_only") is not True or evaluation.get("warnings")
            or evaluation.get("failed_checks") or not evaluation.get("checks")
            or not all(v is True for v in evaluation["checks"].values())):
        raise ValueError("E1 evaluation did not pass all checks without warnings")
    expected_scores = {"accuracy": 0.9508364093300872, "cross_entropy": 0.1460895589143588,
                       "macro_f1": 0.9491439636618202, "weighted_f1": 0.9508945828487836}
    if any(not math.isclose(evaluation[key], value, rel_tol=0, abs_tol=1e-12) for key, value in expected_scores.items()):
        raise ValueError("Evaluation differs from the verified E1 results")
    if evaluation["history_comparison"]["accuracy_difference"] != 0 or evaluation["history_comparison"]["loss_difference"] != 0:
        raise ValueError("Verified evaluation must reproduce epoch 10 accuracy/loss exactly")

    def digest(path):
        with Path(path).open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

    pinned = "14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
    if digest(setup / "transfer_manifest.json") != pinned:
        raise ValueError("Pinned setup manifest changed")
    transfer = json.loads((setup / "transfer_manifest.json").read_text())
    expected_hashes = {**transfer["sha256"], "transfer_manifest.json": pinned}
    inputs = {"checkpoint": run / "best_checkpoint.pt", "configuration": run / "configuration.json",
        "history": run / "history.csv", "diagnostic": working / "mobilenetv2_setup_check.json",
        "setup_report": working / "gpu_setup_check.json", "cache_report": working / "training_image_cache_check.json"}
    inputs.update({name: setup / name for name in expected_hashes})
    before = {name: digest(path) for name, path in inputs.items()}
    if before != evaluation["input_sha256"] or any(before[name] != value for name, value in expected_hashes.items()):
        raise ValueError("Evaluated checkpoint/configuration/source/split/mapping hashes changed")
    inputs["evaluation"] = evaluation_path
    before["evaluation"] = digest(evaluation_path)
    config = json.loads(inputs["configuration"].read_text())
    mapping = json.loads((setup / "data/processed/class_mapping.json").read_text())
    names = [mapping["index_to_class"][str(i)] for i in range(8)]
    checkpoint = torch.load(inputs["checkpoint"], weights_only=True, map_location="cpu")
    if (config["run_name"] != run.name or checkpoint["configuration"] != config or checkpoint["best_epoch"] != 10
            or checkpoint["class_mapping"] != mapping or config["class_mapping"] != mapping
            or evaluation["class_mapping"] != mapping or transfer["classes"] != mapping
            or names != sorted(names) or len(set(names)) != 8
            or mapping["class_to_index"] != dict(zip(names, range(8)))
            or checkpoint["split_checksum"] != before["data/processed/split_manifest.csv"]
            or config["split_sha256"] != checkpoint["split_checksum"]
            or not math.isclose(checkpoint["best_validation_loss"], expected_scores["cross_entropy"], rel_tol=0, abs_tol=1e-12)):
        raise ValueError("E1 checkpoint/configuration/class encoding is inconsistent")
    if before["history"] != "b0bbd7134ca16a70fbb480b8895b5debf97007621de16aaa52ad88ec1a36b3ea":
        raise ValueError("Completed E1 history changed")
    module = sys.modules.get("src.data_pipeline")
    if module is None or Path(module.__file__).resolve() != setup / "src/data_pipeline.py":
        raise RuntimeError("Verified preprocessing/Dataset module is unavailable")
    from src.data_pipeline import reference_preprocess, FacialFeatureDataset
    data_root = Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2/")
    sample_path, test_directory, filenames = _e1_discover_test_input(data_root)
    inputs["sample_submission"] = sample_path
    before["sample_submission"] = digest(sample_path)
    sample = pd.read_csv(sample_path, dtype=str, keep_default_na=False, na_filter=False)
    original_ids, lookup = _e1_test_lookup(sample, filenames)
    # Sample 'class' entries are placeholders and are never consulted.
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None]
    cache = Path(config["training_image_directory"]).resolve()
    cache_report = json.loads(inputs["cache_report"].read_text())
    if (not cache.is_dir() or cache_report["status"] != "passed"
            or Path(cache_report["cache_directory"]).resolve() != cache):
        raise RuntimeError("Existing validation-image cache needed for the preprocessing check is unavailable")
    validation_base = FacialFeatureDataset(setup, "validation", training_image_directory=cache)
    validation = ImageNetNormalizedDataset(validation_base)
    if (validation_base.augmentation is not None or not torch.equal(validation.mean, mean)
            or not torch.equal(validation.std, std) or config["preprocessing"]["normalisation_applications"] != 1):
        raise ValueError("E1 validation normalisation/augmentation changed")
    expected_tensor, _, validation_id = validation[0]
    with Image.open(cache / (validation_id + ".jpg")) as image:
        image.load()
        test_tensor = _e1_test_tensor(image, reference_preprocess, mean, std)
    if not torch.equal(test_tensor, expected_tensor) or not bool(torch.isfinite(test_tensor).all()):
        raise ValueError("Test preprocessing differs from E1 validation preprocessing")
    if not torch.cuda.is_available():
        raise RuntimeError("Required cuda:0 is unavailable")
    device = torch.device("cuda:0")
    model = FrozenMobileNetV2(weights=None).to(dtype=torch.float32)  # No download.
    parameters = {"total": sum(p.numel() for p in model.parameters()),
                  "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
    if parameters != config["parameters"] or parameters != {"total": 2234120, "trainable": 10248}:
        raise ValueError("E1 model architecture changed")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(device).eval()
    dataset = E1TestDataset(test_directory, original_ids, lookup, reference_preprocess, mean, std)
    loader = DataLoader(dataset, batch_size=32, num_workers=0, drop_last=False, shuffle=False)
    predicted_ids, predicted_indices, first_batch_seconds = [], [], []
    rough_projection = None
    torch.cuda.synchronize(device)
    started = time.perf_counter()
    iterator = iter(loader)
    with torch.inference_mode(), torch.autocast(device_type="cuda", enabled=False):
        for batch_number in range(1, len(loader) + 1):
            torch.cuda.synchronize(device)
            batch_started = time.perf_counter()
            images, ids = next(iterator)  # First batches are retained, never rerun.
            start, end = len(predicted_ids), len(predicted_ids) + len(ids)
            if list(ids) != original_ids[start:end] or images.dtype != torch.float32 or images.shape != (len(ids), 3, 224, 224):
                raise ValueError("Test batch order/shape/precision changed")
            images = images.to(device)
            if not bool(torch.isfinite(images).all()):
                raise FloatingPointError("Nonfinite test tensors")
            logits = model(images)
            if logits.shape != (len(ids), 8) or logits.dtype != torch.float32 or not bool(torch.isfinite(logits).all()):
                raise FloatingPointError("Invalid/nonfinite test logits")
            predicted_indices.extend(logits.argmax(dim=1).cpu().tolist())
            predicted_ids.extend(ids)
            torch.cuda.synchronize(device)
            duration = time.perf_counter() - batch_started
            if batch_number <= 3:
                first_batch_seconds.append(duration)
                print(f"Test batch {batch_number}: {duration:.3f}s (input loading included; predictions retained)", flush=True)
            if batch_number == min(3, len(loader)):
                rough_projection = {"label": "Rough projection from first input-mount batches, NOT measured full runtime",
                    "sample_batches": len(first_batch_seconds), "seconds": statistics.mean(first_batch_seconds) * len(loader)}
                print(json.dumps(rough_projection), flush=True)
            if batch_number % 100 == 0 or batch_number == len(loader):
                print(f"Test inference: {len(predicted_ids):,}/{len(original_ids):,} images", flush=True)
    torch.cuda.synchronize(device)
    inference_seconds = time.perf_counter() - started
    submission = _e1_submission_frame(predicted_ids, predicted_indices, mapping["index_to_class"])
    checks = {"successful_e1_evaluation": True, "complete_one_to_one_filename_coverage": True,
        "preprocessing_matches_validation": True, "finite_tensors_and_eight_logits": True,
        "exact_original_id_order": predicted_ids == original_ids,
        "unique_complete_29443_predictions": len(submission) == 29443 and submission["image_id"].is_unique,
        "official_classes_only": bool(submission["class"].isin(names).all()),
        "original_inputs_unchanged": all(digest(path) == before[name] for name, path in inputs.items())}
    if not all(checks.values()):
        raise ValueError("Submission checks failed: " + str(checks))
    with submission_path.open("x", newline="", encoding="utf-8") as stream:
        submission.to_csv(stream, index=False)  # Exact image_id,class columns; no index.
    reread = pd.read_csv(submission_path, dtype=str, keep_default_na=False, na_filter=False)
    checks["saved_csv_roundtrip"] = list(reread.columns) == ["image_id", "class"] and reread.to_dict("list") == submission.to_dict("list")
    if not checks["saved_csv_roundtrip"]:
        raise ValueError("Saved submission failed string-ID/content round-trip; file retained for inspection")
    result = {"status": "passed", "checks": checks, "failed_checks": [], "row_count": len(submission),
        "class_counts": {name: int(submission["class"].eq(name).sum()) for name in names},
        "checkpoint_sha256": before["checkpoint"], "configuration_sha256": before["configuration"],
        "sample_sha256": before["sample_submission"], "submission_sha256": digest(submission_path),
        "input_sha256": before, "test_directory": str(test_directory),
        "id_mapping": "Original sample ID preserved; zfill(5) + '.jpg' for lookup only",
        "preprocessing": config["preprocessing"], "parameters": parameters,
        "device": str(device), "gpu": torch.cuda.get_device_name(0), "dtype": "float32", "amp": False,
        "batch_size": 32, "num_workers": 0, "drop_last": False, "shuffle": False,
        "first_batch_seconds_loading_inclusive": first_batch_seconds, "rough_runtime_projection": rough_projection,
        "inference_seconds_loading_inclusive": inference_seconds, "completed_batches": len(loader),
        "runtime": {"python": platform.python_version(), "torch": str(torch.__version__), "torchvision": torchvision.__version__,
                    "numpy": np.__version__, "pandas": pd.__version__, "Pillow": version("Pillow"), "cuda_build": torch.version.cuda},
        "outputs": [str(submission_path), str(check_path)],
        "limitations": ["Predicted class counts are not hidden-test labels or accuracy.",
                        "Input-mount timings include startup/cache effects; no cached-validation speed assumption.",
                        "No competition upload was performed."]}
    with check_path.open("x", encoding="utf-8") as stream:
        json.dump(result, stream, indent=2, allow_nan=False)
        stream.write("\n")
    print(json.dumps({key: result[key] for key in ["status", "row_count", "class_counts", "inference_seconds_loading_inclusive", "outputs"]}, indent=2))
    return result


if RUN_TEST_PREDICTION:
    submission_E1_check = _predict_e1_test()
else:
    print("RUN_TEST_PREDICTION=False: no test images decoded, predictions or submission files created.")